# CoralGuard: Multimodal Early Warning of Coral Reef Bleaching

**Objective:** Predict coral reef bleaching risk 2-8 weeks ahead per reef site by fusing 3 distinct modalities:
1. **Satellite SST & DHW:** Thermal stress accumulation from NOAA Coral Reef Watch.
2. **Citizen Diver Reports:** Observer-confidence weighted ground truth from Reef Check and CoralWatch.
3. **Underwater Acoustic Soundscapes:** Snapping shrimp (2–20 kHz) and fish chorus (50–1000 Hz) biophony, ACI, ADI, Bioacoustic Index (BI), and NDSI.

---

## 1. Load Aligned Multi-Modal Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

data_path = '../data/processed/full_aligned_dataset.parquet' if os.path.exists('../data/processed/full_aligned_dataset.parquet') else 'data/processed/full_aligned_dataset.parquet'
df = pd.read_parquet(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Unique Reef Sites Monitored: {df['site_id'].nunique()}")
print(f"Temporal Span: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}")
df.head()

## 2. Modality 1: Satellite Thermal Stress Dynamics (SST, SSTA & DHW)

In [ ]:
sample_site = df['site_id'].iloc[0]
site_df = df[df['site_id'] == sample_site].sort_values('date')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(site_df['date'], site_df['sst'], label='SST (°C)', color='#0284C7', lw=1.8)
ax1.plot(site_df['date'], site_df['bleaching_threshold'], label='Bleaching Threshold (MMM+1)', color='#DC2626', linestyle='--', lw=1.5)
ax2.plot(site_df['date'], site_df['dhw'], label='Degree Heating Weeks (DHW)', color='#D97706', lw=1.8)

ax1.set_ylabel('SST (°C)', color='#0284C7', fontsize=12)
ax2.set_ylabel('DHW (°C-weeks)', color='#D97706', fontsize=12)
ax1.set_title(f'Satellite Thermal Stress Accumulation: {sample_site}', fontsize=14, pad=12)
plt.show()

## 3. Modality 2: Citizen Diver Reports & Spatial Density

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df['diver_reports_count_50km_30d'], bins=25, kde=True, ax=ax1, color='#EA580C')
ax1.set_title('30-day Diver Report Volume within 50km', fontsize=12)
ax1.set_xlabel('Number of Diver Reports')

sns.boxplot(data=df, x='target_t14', y='diver_weighted_severity_30d', ax=ax2, palette='YlOrRd')
ax2.set_title('Weighted Diver Bleaching Severity by True t+14 Class', fontsize=12)
ax2.set_xticklabels(['None', 'Mild', 'Severe'])
ax2.set_xlabel('Future Bleaching Severity (t+14d)')
plt.show()

## 4. Modality 3: Underwater Acoustic Soundscapes

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 9))

sample_sub = df.sample(min(1000, len(df)), random_state=42)

# 1. Snapping shrimp vs Fish chorus
sns.scatterplot(data=sample_sub, x='fish_chorus_energy_ratio', y='shrimp_band_energy_ratio',
                hue='target_t14', palette='Spectral', ax=ax1, alpha=0.7)
ax1.set_title('Biophony Signature: Snapping Shrimp vs Fish Chorus', fontsize=12)
ax1.set_xlabel('Fish Chorus Energy Ratio (50-1000 Hz)')
ax1.set_ylabel('Snapping Shrimp Energy Ratio (2-20 kHz)')

# 2. Acoustic Complexity Index
sns.kdeplot(data=df[df['target_t14'] == 0]['aci'], label='Healthy Reef (None)', ax=ax2, fill=True, color='green')
sns.kdeplot(data=df[df['target_t14'] == 2]['aci'], label='Degraded Reef (Severe)', ax=ax2, fill=True, color='red')
ax2.set_title('Acoustic Complexity Index (ACI) Distribution', fontsize=12)
ax2.set_xlabel('ACI Value')
ax2.legend()

# 3. Bioacoustic Index (BI)
sns.boxplot(data=df, x='target_t14', y='bioacoustic_index', ax=ax3, palette='viridis')
ax3.set_title('Bioacoustic Index (2-8 kHz Biophony Area) by Future Bleaching Class', fontsize=12)
ax3.set_xticklabels(['None', 'Mild', 'Severe'])
ax3.set_xlabel('Future Bleaching Severity (t+14d)')

# 4. Normalized Difference Soundscape Index (NDSI)
sns.kdeplot(data=df[df['target_t14'] == 0]['ndsi'], label='Healthy Reef (None)', ax=ax4, fill=True, color='teal')
sns.kdeplot(data=df[df['target_t14'] == 2]['ndsi'], label='Degraded Reef (Severe)', ax=ax4, fill=True, color='crimson')
ax4.set_title('NDSI: Biophony vs Anthrophony Partitioning', fontsize=12)
ax4.set_xlabel('NDSI [-1.0 to +1.0]')
ax4.legend()

plt.tight_layout()
plt.show()

## 5. Model Ablation & Performance Benchmark

In [ ]:
table_path = '../reports/ablation_metrics_table.csv' if os.path.exists('../reports/ablation_metrics_table.csv') else 'reports/ablation_metrics_table.csv'
ablation_df = pd.read_csv(table_path)
ablation_df

## 6. Early Warning False Alarm Rate (FAR) Reduction

In [ ]:
import json
far_path = '../reports/early_warning_far_comparison.json' if os.path.exists('../reports/early_warning_far_comparison.json') else 'reports/early_warning_far_comparison.json'
with open(far_path, 'r') as f:
    far_comp = json.load(f)

print(json.dumps(far_comp, indent=2))

# Bar chart comparison
far_data = pd.DataFrame([
    {'Model': 'NOAA DHW Baseline', 'Metric': 'Severe Recall', 'Value': far_comp['noaa_dhw_baseline']['severe_recall'] * 100},
    {'Model': 'CoralGuard Multimodal', 'Metric': 'Severe Recall', 'Value': far_comp['multimodal_early_warning']['severe_recall'] * 100},
    {'Model': 'NOAA DHW Baseline', 'Metric': 'False Alarm Rate', 'Value': far_comp['noaa_dhw_baseline']['false_alarm_rate'] * 100},
    {'Model': 'CoralGuard Multimodal', 'Metric': 'False Alarm Rate', 'Value': far_comp['multimodal_early_warning']['false_alarm_rate'] * 100},
])

plt.figure(figsize=(8, 4))
sns.barplot(data=far_data, x='Metric', y='Value', hue='Model', palette='Set2')
plt.title('Operational Early Warning Comparison (+14d Ahead)')
plt.ylabel('Percentage (%)')
plt.show()